# Appendix D: Metabolic Time Scaling (Kleiber's Law)

Executable verification code for all calculations in Appendix D.
Open in Jupyter to modify parameters or rerun.

## Section I: Metabolic Scaling Regression (7 species)

Linear regression of log10(heart period) vs log10(body mass) for
7 well-documented species spanning 6 orders of magnitude in mass.

In [ ]:
import numpy as np
from scipy import stats

# 7-species heart-period data: (log10_M, log10_T)
log_M = np.array([-2.74, -1.60, -0.52, 0.60, 1.85, 3.60, 5.18])
log_T = np.array([-1.00, -1.00, -0.77, -0.30, -0.08, 0.30, 0.82])

slope, intercept, r, p, se = stats.linregress(log_M, log_T)
n = len(log_M)
dof = n - 2
t_crit = stats.t.ppf(0.975, dof)
ci_lo = slope - t_crit * se
ci_hi = slope + t_crit * se

pred = intercept + slope * log_M
resid = log_T - pred
ss_res = np.sum(resid**2)
ss_tot = np.sum((log_T - np.mean(log_T))**2)
r2 = r**2
r2_adj = 1 - (1 - r2) * (n - 1) / dof
rmse = np.sqrt(ss_res / dof)
f_stat = (ss_tot - ss_res) / (ss_res / dof)

print(f"Slope (n):          {slope:.4f} +/- {se:.4f}")
print(f"Intercept (a):      {intercept:.4f}")
print(f"95% CI for n:       [{ci_lo:.3f}, {ci_hi:.3f}]")
print(f"R-squared:          {r2:.4f}")
print(f"Adjusted R-squared: {r2_adj:.4f}")
print(f"RMSE:               {rmse:.4f}")
print(f"F-statistic:        {f_stat:.1f}")
print(f"p-value:            {p:.2e}")

## Section II: Residual Diagnostics

Shapiro-Wilk normality test and Breusch-Pagan heteroscedasticity test
on the 7-species regression residuals.

In [ ]:
import numpy as np
from scipy import stats

# Reproduce residual diagnostics from 7-species regression
log_M = np.array([-2.74, -1.60, -0.52, 0.60, 1.85, 3.60, 5.18])
log_T = np.array([-1.00, -1.00, -0.77, -0.30, -0.08, 0.30, 0.82])

slope, intercept, r, p, se = stats.linregress(log_M, log_T)
pred = intercept + slope * log_M
resid = log_T - pred
n = len(log_M)
dof = n - 2
rmse = np.sqrt(np.sum(resid**2) / dof)

# Shapiro-Wilk normality test
sw_W, sw_p = stats.shapiro(resid)
print(f"Shapiro-Wilk W = {sw_W:.3f}, p = {sw_p:.3f}")

# Breusch-Pagan heteroscedasticity test
resid_sq = resid**2
bp_sl, bp_int, bp_r, bp_p, bp_se = stats.linregress(log_M, resid_sq)
bp_pred = bp_int + bp_sl * log_M
bp_ss_reg = np.sum((bp_pred - np.mean(resid_sq))**2)
bp_ss_tot = np.sum((resid_sq - np.mean(resid_sq))**2)
bp_r2 = bp_ss_reg / bp_ss_tot if bp_ss_tot > 0 else 0
bp_stat = n * bp_r2
bp_p_val = 1 - stats.chi2.cdf(bp_stat, df=1)
print(f"Breusch-Pagan stat = {bp_stat:.3f}, p = {bp_p_val:.3f}")
print(f"Conclusion: {'No' if bp_p_val > 0.05 else ''} significant heteroscedasticity")

## Section III: AnAge Database Analysis (Bin-Median Method)

Scale-balanced bin-median regression across 8 mass bins
spanning 10^-9 to 10^7 kg, yielding the primary empirical exponent
of ~0.21.

In [ ]:
import numpy as np
from scipy import stats

# 2-dex bin medians: (bin_center_log10_M, median_log10_T_max)
bin_x = np.array([-8.0, -6.0, -4.0, -2.0, 0.0, 2.0, 4.0, 6.0])
bin_y = np.array([-1.5, -0.5,  0.0,  0.4, 1.0, 1.5, 1.8, 2.2])

slope, intercept, r, p, se = stats.linregress(bin_x, bin_y)
r2 = r**2
print(f"Bin-median slope n = {slope:.4f}")
print(f"Bin-median R^2     = {r2:.4f}")
print(f"Intercept          = {intercept:.4f}")
print(f"Std error          = {se:.4f}")

## Section IV: Comprehensive Verification Code

Complete verification of all Appendix D calculations with error
propagation. Split into sub-cells by analysis stage.

### Part 1: Biological Data

In [ ]:
#!/usr/bin/env python3
"""
Appendix D: Metabolic Time Scaling Verification
Complete code for all calculations with error propagation
"""

import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# PART 1: BIOLOGICAL DATA
# =============================================================================

# Species data: [name, mass_kg, mass_err, period_s, period_err]
biological_data = [
    ("Etruscan Shrew", 1.8e-3, 0.2e-3, 0.10, 0.01),
    ("Mouse", 2.5e-2, 0.5e-2, 0.10, 0.01),
    ("Rat", 3.0e-1, 0.5e-1, 0.17, 0.02),
    ("Cat", 4.0e0, 0.5e0, 0.50, 0.05),
    ("Human", 7.0e1, 5.0e0, 0.83, 0.10),
    ("Elephant", 4.0e3, 0.5e3, 2.00, 0.20),
    ("Blue Whale", 1.5e5, 0.3e5, 6.67, 1.00),
]

# Extract arrays
names = [d[0] for d in biological_data]
masses = np.array([d[1] for d in biological_data])
mass_errs = np.array([d[2] for d in biological_data])
periods = np.array([d[3] for d in biological_data])
period_errs = np.array([d[4] for d in biological_data])

# Log transform
log_masses = np.log10(masses)
log_periods = np.log10(periods)

# Error propagation for log transform: delta_log(x) = delta_x / (x * ln(10))
log_mass_errs = mass_errs / (masses * np.log(10))
log_period_errs = period_errs / (periods * np.log(10))

print("=" * 70)
print("BIOLOGICAL TIME SCALING ANALYSIS")
print("=" * 70)
print("\nData Table:")
print(f"{'Species':<20} {'Mass (kg)':<12} {'log10(M)':<10} {'Period (s)':<12} {'log10(T)':<10}")
print("-" * 70)
for i, name in enumerate(names):
    print(f"{name:<20} {masses[i]:<12.2e} {log_masses[i]:<10.2f} {periods[i]:<12.2f} {log_periods[i]:<10.2f}")

### Part 2: Linear Regression

In [ ]:
# =============================================================================
# PART 2: LINEAR REGRESSION
# =============================================================================

print("\n" + "=" * 70)
print("LINEAR REGRESSION: log10(T) = a + n * log10(M)")
print("=" * 70)

# Perform regression
slope, intercept, r_value, p_value, std_err = stats.linregress(log_masses, log_periods)

# Calculate additional statistics
n_points = len(log_masses)
dof = n_points - 2  # degrees of freedom

# Calculate confidence interval for slope
t_critical = stats.t.ppf(0.975, dof)  # 95% CI
slope_ci_low = slope - t_critical * std_err
slope_ci_high = slope + t_critical * std_err

# Calculate predictions and residuals
predictions = intercept + slope * log_masses
residuals = log_periods - predictions
ss_res = np.sum(residuals**2)
ss_tot = np.sum((log_periods - np.mean(log_periods))**2)
rmse = np.sqrt(ss_res / dof)

# F-statistic
f_stat = (ss_tot - ss_res) / (ss_res / dof)
f_p_value = 1 - stats.f.cdf(f_stat, 1, dof)

print(f"\nRegression Results:")
print(f"  Slope (n):           {slope:.4f} +/- {std_err:.4f}")
print(f"  Intercept (a):       {intercept:.4f}")
print(f"  95% CI for slope:    [{slope_ci_low:.3f}, {slope_ci_high:.3f}]")
print(f"\nGoodness of Fit:")
print(f"  R-squared:           {r_value**2:.4f}")
print(f"  Adjusted R-squared:  {1 - (1-r_value**2)*(n_points-1)/dof:.4f}")
print(f"  RMSE:                {rmse:.4f}")
print(f"  F-statistic:         {f_stat:.1f}")
print(f"  p-value:             {p_value:.2e}")

### Part 3: Residual Analysis

In [ ]:
# =============================================================================
# PART 3: RESIDUAL ANALYSIS
# =============================================================================

print("\n" + "=" * 70)
print("RESIDUAL ANALYSIS")
print("=" * 70)

print(f"\n{'Species':<20} {'Observed':<10} {'Predicted':<10} {'Residual':<10} {'Std Res':<10}")
print("-" * 60)
std_residuals = residuals / rmse
for i, name in enumerate(names):
    print(f"{name:<20} {log_periods[i]:<10.3f} {predictions[i]:<10.3f} {residuals[i]:<+10.3f} {std_residuals[i]:<+10.2f}")

# Shapiro-Wilk test for normality
shapiro_stat, shapiro_p = stats.shapiro(residuals)
print(f"\nShapiro-Wilk Test for Normality:")
print(f"  W-statistic: {shapiro_stat:.3f}")
print(f"  p-value: {shapiro_p:.3f}")
print(f"  Interpretation: Residuals {'are' if shapiro_p > 0.05 else 'are NOT'} normally distributed (alpha=0.05)")

### Part 4: Comparison with Theoretical Value

In [ ]:
# =============================================================================
# PART 4: COMPARISON WITH THEORETICAL VALUE
# =============================================================================

print("\n" + "=" * 70)
print("COMPARISON WITH THEORETICAL n = 0.25")
print("=" * 70)

theoretical_n = 0.25
z_score = (slope - theoretical_n) / std_err
z_p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))

print(f"\nTheoretical exponent (from Kleiber): {theoretical_n:.3f}")
print(f"Observed exponent:                   {slope:.3f}")
print(f"Difference:                          {slope - theoretical_n:+.3f}")
print(f"Z-score:                             {z_score:.2f}")
print(f"Two-tailed p-value:                  {z_p_value:.3f}")
print(f"\nConclusion: Observed exponent is {'NOT ' if z_p_value >= 0.05 else ''}significantly different from 0.25")

### Part 5: City Scaling Verification

In [ ]:
# =============================================================================
# PART 5: CITY SCALING VERIFICATION
# =============================================================================

print("\n" + "=" * 70)
print("CITY SCALING VERIFICATION")
print("=" * 70)

# City data: [name, population_millions, energy_TWh]
city_data = [
    ("Houston", 2.3, 142),
    ("New York", 8.3, 380),
    ("Los Angeles", 4.0, 245),
    ("Chicago", 2.7, 168),
    ("Tokyo", 13.5, 456),
]

city_names = [d[0] for d in city_data]
populations = np.array([d[1] * 1e6 for d in city_data])  # Convert to actual population
energies = np.array([d[2] for d in city_data])

log_pop = np.log10(populations)
log_energy = np.log10(energies)

# Regression
slope_city, intercept_city, r_city, p_city, stderr_city = stats.linregress(log_pop, log_energy)

print(f"\nCity Energy Scaling: log10(E) = a + n * log10(Population)")
print(f"  Slope (n):           {slope_city:.3f} +/- {stderr_city:.3f}")
print(f"  R-squared:           {r_city**2:.3f}")
print(f"  Theoretical (West):  0.85")
print(f"  Within error bars:   {'Yes' if abs(slope_city - 0.85) < 2*stderr_city else 'No'}")

# Verify 2x population efficiency claim
print(f"\n2x Population Energy Efficiency Verification:")
print(f"  If Pop_B = 2 * Pop_A:")
print(f"  E_B / E_A = 2^{slope_city:.2f} = {2**slope_city:.2f}")
print(f"  Per capita ratio = {2**slope_city / 2:.2f}")
print(f"  Claim: Larger city is {(1 - 2**slope_city/2)*100:.1f}% more efficient per capita")

### Part 6: Exponent Sensitivity Analysis

In [ ]:
# =============================================================================
# PART 6: SENSITIVITY ANALYSIS - EXPONENT VARIATION
# =============================================================================

print("\n" + "=" * 70)
print("SENSITIVITY ANALYSIS: EFFECT OF EXPONENT VARIATION")
print("=" * 70)

# 0.20-0.22 is the primary empirical value (full AnAge dataset, scale-balanced bin medians)
# 0.25 is the WBE theoretical prediction
# 0.28 included for sensitivity analysis only; NOT the primary value
# 0.30 is the upper bound of the biological confidence interval
exponents = [0.20, 0.25, 0.28, 0.30]
test_masses = [1e0, 1e10, 1e30, 1e50]  # kg

print(f"\nTime scaling ratio T/T_reference for different exponents:")
print(f"{'Mass (kg)':<15}", end="")
for n in exponents:
    print(f"{'n='+str(n):<12}", end="")
print()
print("-" * 63)

for m in test_masses:
    print(f"{m:<15.0e}", end="")
    for n in exponents:
        ratio = m**n
        print(f"{ratio:<12.2e}", end="")
    print()

print(f"\nAt M = 10^42 kg (galaxy):")
print(f"  Using n=0.21 (primary, full AnAge): T ratio = {(1e42)**0.21:.2e}")
print(f"  Using n=0.25: T ratio = {(1e42)**0.25:.2e}")
print(f"  Using n=0.28 (illustrative only): T ratio = {(1e42)**0.28:.2e}")
print(f"  Primary/illustrative factor: {(1e42)**0.28 / (1e42)**0.21:.1f}x")

### Part 7: Cosmic Time Scaling (Speculative)

In [ ]:
# =============================================================================
# PART 7: COSMIC TIME SCALING (SPECULATIVE)
# =============================================================================

print("\n" + "=" * 70)
print("COSMIC TIME SCALING (SPECULATIVE)")
print("=" * 70)
print("\nWARNING: These calculations extend beyond verified biology!")
print("Results should be interpreted as exploratory, not definitive.\n")

# Reference values
T_human = 3.0  # seconds (conscious moment)
M_human = 70   # kg

# Structures to analyze
cosmic_structures = [
    ("Bacterium", 1e-15),
    ("Human", 70),
    ("Nation", 1e12),
    ("Biosphere", 1e15),
    ("Sun", 2e30),
    ("Galaxy", 1e42),
    ("Universe", 1e53),
]

print(f"Using T proportional to M^0.25 (biological scaling):")
print(f"Reference: Human (M={M_human} kg, T_char={T_human} s)\n")

print(f"{'Structure':<15} {'Mass (kg)':<12} {'T_char (s)':<12} {'In Years':<15} {'Interpretation':<30}")
print("-" * 84)

for name, mass in cosmic_structures:
    # Calculate characteristic time
    T_char = T_human * (mass / M_human)**0.25

    # Convert to years if large
    T_years = T_char / (365.25 * 24 * 3600)

    if T_char < 1:
        interpretation = f"{T_char*1000:.1f} milliseconds"
    elif T_char < 3600:
        interpretation = f"{T_char:.1f} seconds"
    elif T_char < 86400 * 365:
        interpretation = f"{T_char/86400:.1f} days"
    elif T_years < 1e6:
        interpretation = f"{T_years:.0f} years"
    elif T_years < 1e9:
        interpretation = f"{T_years/1e6:.1f} million years"
    elif T_years < 1e12:
        interpretation = f"{T_years/1e9:.1f} billion years"
    else:
        interpretation = f"{T_years/1e12:.1f} trillion years"

    print(f"{name:<15} {mass:<12.0e} {T_char:<12.2e} {T_years:<15.2e} {interpretation:<30}")

# Universe age calculation
print(f"\nUniverse 'Subjective Age' Calculation:")
universe_age_years = 13.8e9
universe_mass = 1e53
human_mass = 70

time_dilation = (universe_mass / human_mass)**0.25
subjective_age = universe_age_years / time_dilation
subjective_hours = subjective_age * 365.25 * 24

print(f"  Universe age: {universe_age_years:.2e} years")
print(f"  Time dilation factor: {time_dilation:.2e}")
print(f"  Subjective age: {subjective_age:.2e} years = {subjective_hours:.1f} hours")
print(f"  Subjective age: {subjective_hours:.1f} hours (order of magnitude: ~20 hours)")

### Part 8: Full AnAge Bin-Median Analysis Simulation

In [ ]:
# =============================================================================
# PART 8: FULL ANAGE BIN-MEDIAN ANALYSIS SIMULATION
# =============================================================================

print("\n" + "=" * 70)
print("FULL AnAge BIN-MEDIAN ANALYSIS SIMULATION")
print("=" * 70)
print("\nSimulating scale-balanced bin-median methodology (Section 2.5)")
print("Using synthetic data calibrated to AnAge database characteristics\n")

np.random.seed(42)

# Define 8 mass bins spanning 10^-9 to 10^7 kg (2-dex each)
# Each bin: (label, log10_M_center, N_species, scatter, skew_parameter)
#
# True underlying trend: log10(lifespan) = 0.205 * log10(mass) + 0.50
# This yields bin-median slope ~0.21 (the primary empirical exponent).
#
# IMPORTANT: In real AnAge data, pooled OLS gives slope ~0.14 (not ~0.24),
# because densely-sampled mass ranges (rodents, small mammals at 0.01-10 kg)
# dominate the regression, and those ranges have LOW local slopes.
# Bin-medians RAISE the slope to ~0.20-0.22 by giving each mass decade
# equal weight. This simulation demonstrates the bin-median methodology
# but does NOT replicate the real OLS-vs-bin-median direction.
#
# The skew parameter here adds longevity outliers to mammalian bins,
# inflating the simulated OLS above the true slope. In reality, the
# pooled OLS bias goes the OTHER direction (downward, not upward).
# We subtract skew * ln(2) so the MEDIAN stays on the 0.205 trend line.

true_slope = 0.205
true_intercept = 0.50

bin_definitions = [
    ("Nematodes/small inverts",  -8.0,  50,  0.30, 0.00),  # 10^-9 to 10^-7
    ("Insects (small)",          -6.0,  30,  0.30, 0.00),  # 10^-7 to 10^-5
    ("Insects/small vertebrates",-4.0, 100,  0.30, 0.00),  # 10^-5 to 10^-3
    ("Small inverts/rodents",    -2.0, 150,  0.30, 0.10),  # 10^-3 to 10^-1
    ("Small mammals",             0.0, 300,  0.35, 0.55),  # 10^-1 to 10^1
    ("Medium mammals",            2.0, 250,  0.35, 0.80),  # 10^1 to 10^3
    ("Large mammals",             4.0,  80,  0.40, 1.05),  # 10^3 to 10^5
    ("Megafauna/cetaceans",       6.0,  15,  0.40, 1.25),  # 10^5 to 10^7
]

# Generate synthetic species data for each bin
all_log_masses = []
all_log_lifespans = []

for label, center, n_species, scatter, skew in bin_definitions:
    # Spread masses uniformly within the 2-dex bin
    log_m = np.random.uniform(center - 1.0, center + 1.0, n_species)
    # Expected lifespan on the true 0.205 trend line
    expected_log_life = true_slope * log_m + true_intercept
    # Symmetric scatter
    noise = np.random.normal(0, scatter, n_species)
    # Add right-skewed tail for mammalian bins (simulates longevity outliers)
    if skew > 0:
        skew_component = np.random.exponential(skew, n_species)
        # Subtract median of exponential so bin MEDIAN stays on trend
        noise = noise + skew_component - skew * np.log(2)
    log_life = expected_log_life + noise
    all_log_masses.extend(log_m)
    all_log_lifespans.extend(log_life)

all_log_masses = np.array(all_log_masses)
all_log_lifespans = np.array(all_log_lifespans)

# --- Raw OLS on all species ---
slope_raw, intercept_raw, r_raw, p_raw, se_raw = stats.linregress(
    all_log_masses, all_log_lifespans
)

print(f"Total synthetic species: {len(all_log_masses)}")
print(f"\n--- Method 1: Raw OLS on all {len(all_log_masses)} species ---")
print(f"  Slope (exponent):    {slope_raw:.4f} +/- {se_raw:.4f}")
print(f"  Intercept:           {intercept_raw:.4f}")
print(f"  R-squared:           {r_raw**2:.4f}")
print(f"  Expected range:      ~0.24 (inflated by skewed noise; real AnAge pooled OLS is ~0.14)")

# --- Bin-median methodology ---
bin_centers = []
bin_medians = []
print(f"\n--- Method 2: Scale-balanced bin medians ---")
print(f"\n  {'Bin Label':<28} {'log10(M) ctr':<14} {'N species':<11} {'Median log10(T)':<16}")
print("  " + "-" * 69)

for label, center, n_species, scatter, skew in bin_definitions:
    mask = (all_log_masses >= center - 1.0) & (all_log_masses < center + 1.0)
    bin_lifespans = all_log_lifespans[mask]
    median_val = np.median(bin_lifespans)
    bin_centers.append(center)
    bin_medians.append(median_val)
    print(f"  {label:<28} {center:<14.1f} {np.sum(mask):<11d} {median_val:<16.3f}")

bin_centers = np.array(bin_centers)
bin_medians = np.array(bin_medians)

slope_bm, intercept_bm, r_bm, p_bm, se_bm = stats.linregress(
    bin_centers, bin_medians
)

print(f"\n  Bin-median regression:")
print(f"    Slope (exponent):    {slope_bm:.4f} +/- {se_bm:.4f}")
print(f"    Intercept:           {intercept_bm:.4f}")
print(f"    R-squared:           {r_bm**2:.4f}")
print(f"    Expected range:      0.20-0.22 (scale-balanced)")

# --- Compare with 7-species dataset ---
print(f"\n--- Comparison of methods ---")
print(f"  Raw OLS (all species):       n = {slope_raw:.4f}")
print(f"  Scale-balanced bin medians:   n = {slope_bm:.4f}")
print(f"  7-species curated (Part 2):   n = 0.241")
print(f"  Exponent drop (raw -> bin):   {slope_raw - slope_bm:+.4f}")
print(f"\n  Key insight: In real AnAge data, pooled OLS yields ~0.14 because")
print(f"  densely-sampled mass ranges (rodents, small mammals) dominate.")
print(f"  The bin-median method gives each mass decade equal weight,")
print(f"  RAISING the exponent to the PRIMARY empirical value of ~0.21.")

### Part 9: Cosmic Extrapolation Failure at Primary Exponent

In [ ]:
# =============================================================================
# PART 9: COSMIC EXTRAPOLATION FAILURE AT PRIMARY EXPONENT
# =============================================================================

print("\n" + "=" * 70)
print("COSMIC EXTRAPOLATION FAILURE AT PRIMARY EXPONENT")
print("=" * 70)
print("\nReference: Human (M = 70 kg, T_char = 3 s conscious moment)")
print("Using lifespan scaling: T_predicted = T_ref * (M / M_ref)^n\n")

# Reference values
M_ref = 70.0        # kg (human)
T_ref_s = 3.0       # seconds (conscious moment)

# For lifespan extrapolation, use human lifespan as reference
T_ref_life_yr = 122.5  # years (approximate max human lifespan for scaling)
T_ref_life_s = T_ref_life_yr * 365.25 * 24 * 3600  # in seconds

# Cosmic structures with observed characteristic times
cosmic_targets = [
    ("Sun",                2e30,  1e10,    "Main-sequence lifetime (yr)"),
    ("Milky Way",          1.5e42, 1.3e10, "Age of Milky Way (yr)"),
    ("Observable Universe", 1.5e53, 1.37e10, "Age of universe (yr)"),
]

exponents_test = [0.21, 0.25, 0.28]

# Print header
print(f"{'Structure':<22} {'Mass (kg)':<12} {'Observed (yr)':<14} ", end="")
for n in exponents_test:
    print(f"{'n='+str(n)+' (yr)':<16} ", end="")
print()
print("-" * 100)

results = {}
for name, mass, T_obs_yr, description in cosmic_targets:
    print(f"{name:<22} {mass:<12.1e} {T_obs_yr:<14.2e} ", end="")
    row_results = {}
    for n in exponents_test:
        T_pred_yr = T_ref_life_yr * (mass / M_ref)**n
        row_results[n] = T_pred_yr
        print(f"{T_pred_yr:<16.2e} ", end="")
    results[name] = (T_obs_yr, row_results)
    print()

# PASS/FAIL table (within 1 order of magnitude = PASS)
print(f"\n--- PASS/FAIL Assessment (within 1 order of magnitude) ---\n")
print(f"{'Structure':<22} ", end="")
for n in exponents_test:
    print(f"{'n='+str(n):<16} ", end="")
print()
print("-" * 70)

for name, mass, T_obs_yr, description in cosmic_targets:
    T_obs = T_obs_yr
    print(f"{name:<22} ", end="")
    for n in exponents_test:
        T_pred = results[name][1][n]
        log_ratio = np.log10(T_pred / T_obs)
        if abs(log_ratio) <= 1.0:
            status = f"PASS ({log_ratio:+.1f} dex)"
        else:
            status = f"FAIL ({log_ratio:+.1f} dex)"
        print(f"{status:<16} ", end="")
    print()

# Detailed breakdown for the Sun
print(f"\n--- Detailed Sun analysis ---")
sun_mass = 2e30
sun_lifetime_yr = 1e10  # ~10 billion years
for n in exponents_test:
    T_pred = T_ref_life_yr * (sun_mass / M_ref)**n
    ratio = T_pred / sun_lifetime_yr
    log_err = np.log10(ratio)
    print(f"  n = {n:.2f}: T_pred = {T_pred:.2e} yr, "
          f"T_obs = {sun_lifetime_yr:.2e} yr, "
          f"ratio = {ratio:.2e}, "
          f"log10 error = {log_err:+.2f} dex")

print(f"\n--- Key finding ---")
print(f"  At primary exponent n=0.21:")
sun_pred_021 = T_ref_life_yr * (2e30 / 70)**0.21
print(f"    Sun prediction: {sun_pred_021:.2e} yr (actual: ~10^10 yr)")
print(f"    Shortfall: {np.log10(1e10 / sun_pred_021):.1f} orders of magnitude")
print(f"  At illustrative n=0.28:")
sun_pred_028 = T_ref_life_yr * (2e30 / 70)**0.28
print(f"    Sun prediction: {sun_pred_028:.2e} yr (actual: ~10^10 yr)")
print(f"    Error: {np.log10(sun_pred_028 / 1e10):+.1f} orders of magnitude")
print(f"\n  CONCLUSION: The primary biological exponent (0.20-0.22) predicts")
print(f"  stellar lifetimes 2-3 orders of magnitude too short.")
print(f"  This DOES NOT support direct biological-to-cosmic scaling.")
print(f"  The closer fit at n=0.28 is sensitive to species selection")
print(f"  and may represent numerical coincidence.")

print("\n" + "=" * 70)
print("VERIFICATION COMPLETE")
print("=" * 70)